In [0]:
dbutils.widgets.text("task_failed", "false")
dbutils.widgets.text("qa_flag", "false")
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_ref")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
task_failed = dbutils.widgets.get("task_failed").lower() == "true"
qa_flag = dbutils.widgets.get("qa_flag").lower() == "true"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, count, lit, current_timestamp
from datetime import datetime

if not (task_failed and qa_flag):
    dbutils.notebook.exit("Skipping QA checks: task_failed is false or qa_flag is false")


failed_statuses = ["NULL_VALUES_FOUND", "Mismatch", "EMPTY_FILE", "CORRUPT", "MISSING"]

# Read reconciliation log table dynamically from widget
ctl_log_df = spark.table(reconciliation_table)

ctl_candidate_df = (
    ctl_log_df
    .filter(
        col("Status").isin(failed_statuses)
    )
    .orderBy(col("Processed_Timestamp").desc())
    .limit(1)
)

ctl_row = ctl_candidate_df.collect()
if not ctl_row:
    dbutils.notebook.exit("⚠️ No failed CTL file found for QA checks in table: {reconciliation_table}")

ctl_file_name = ctl_row[0]['CTL_File']
ctl_hash = ctl_row[0]['CTL_Hash']

print(f"📄 Selected CTL file for QA: {ctl_file_name}, Hash: {ctl_hash}")